In [ ]:
# === Step 8: Create datasets ===
train_dataset = TimeAwareTokenDataset(df_train, seq_len)
test_dataset = TimeAwareTokenDataset(df_test, seq_len)

# === Step 9: DataLoaders ===
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1)

# === Step 10: Define LSTM model ===
class LSTMModel(nn.Module):
    def __init__(self, input_size=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=128, num_layers=2, batch_first=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

# === Step 11: Train the model ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(140):  # train for 3 epochs
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device).unsqueeze(-1)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

# === Step 12: Predict on test set ===
model.eval()
true_vals, preds = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        pred = model(x).cpu().item()
        y_true = y.item()
        preds.append(scaler_tokens.inverse_transform([[pred]])[0][0])
        true_vals.append(scaler_tokens.inverse_transform([[y_true]])[0][0])

# === Step 13: Print and plot ===
for i in range(len(true_vals)):
    print(f"Step {i+1}: Actual = {round(true_vals[i])}, Predicted = {round(preds[i])}")

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(true_vals, label='Actual', marker='o')
plt.plot(preds, label='Predicted', marker='x')
plt.title("Token Prediction (Last 6 Hours)")
plt.xlabel("30-min Step")
plt.ylabel("Token Count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()